In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
awsaf49_asvpoof_2019_dataset_path = kagglehub.dataset_download('awsaf49/asvpoof-2019-dataset')

print('Data source import complete.')


100%|██████████| 23.6G/23.6G [04:16<00:00, 98.9MB/s]

Extracting files...


Data source import complete.


In [2]:
awsaf49_asvpoof_2019_dataset_path

'/root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1'

In [4]:
import os
import torch
import librosa
import numpy as np
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperModel

DATASET_ROOT = awsaf49_asvpoof_2019_dataset_path + "/LA/LA"
PROTOCOL_FILE = os.path.join(DATASET_ROOT, 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019.LA.cm.train.trn.txt')
AUDIO_DIR = os.path.join(DATASET_ROOT, 'ASVspoof2019_LA_train', 'flac')

OUTPUT_FEATURES = 'X_train_whisper.npy'
OUTPUT_LABELS = 'y_train_whisper.npy'

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"    : {device.upper()}")

model_name = "openai/whisper-base"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperModel.from_pretrained(model_name).to(device)
model.eval()

labels_dict = {}
with open(PROTOCOL_FILE, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 5:
            file_name = parts[1]
            label_str = parts[4]
            labels_dict[file_name] = 0 if label_str == 'bonafide' else 1

print(f"   {len(labels_dict)}    .")

features_list = []
labels_list = []

# file_names_to_process = list(labels_dict.keys())[:1000]
file_names_to_process = list(labels_dict.keys())

print("      Whisper...")
for file_name in tqdm(file_names_to_process):
    audio_path = os.path.join(AUDIO_DIR, f"{file_name}.flac")

    if not os.path.exists(audio_path):
        continue

    audio_array, sampling_rate = librosa.load(audio_path, sr=16000)

    inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    with torch.no_grad():
        encoder_outputs = model.encoder(input_features)
        hidden_states = encoder_outputs.last_hidden_state

        audio_embedding = hidden_states.mean(dim=1).squeeze().cpu().numpy()

    features_list.append(audio_embedding)
    labels_list.append(labels_dict[file_name])

X = np.array(features_list)
y = np.array(labels_list)

np.save(OUTPUT_FEATURES, X)
np.save(OUTPUT_LABELS, y)

print("=============")
print(f"    (X): {X.shape} -> ")
print(f"    (y): {y.shape}")
print(f"   : {OUTPUT_FEATURES}****{OUTPUT_LABELS}")

<>:67: SyntaxWarning: invalid escape sequence '\!'
<>:67: SyntaxWarning: invalid escape sequence '\!'
/tmp/ipykernel_499/2574098353.py:67: SyntaxWarning: invalid escape sequence '\!'
  print("\!")


    : CUDA


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

   25380    .
      Whisper...


100%|██████████| 25380/25380 [18:28<00:00, 22.90it/s]

\!
    (X): (25380, 512) -> 
    (y): (25380,)
   : X_train_whisper.npy****y_train_whisper.npy


In [5]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix

# 1. Load the extracted features and labels
print("Loading extracted features and labels...")
X = np.load('X_train_whisper.npy')
y = np.load('y_train_whisper.npy')

print(f"Dataset Shape: X={X.shape}, y={y.shape}")
count_bonafide = np.sum(y == 0)
count_spoof = np.sum(y == 1)
print(f"Class 0 (Bonafide): {count_bonafide}")
print(f"Class 1 (Spoof): {count_spoof}")

# 2. Stratified Train-Test Split (70% Train, 30% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 3. Handle Imbalance: Calculate scale_pos_weight
# weight = count of negative class / count of positive class
# Since 1 (Spoof) is the majority, we scale it down to balance the importance
scale_weight = count_bonafide / count_spoof

# 4. Initialize XGBoost Classifier
clf = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale_weight,
    random_state=42,
    eval_metric="auc",
    early_stopping_rounds=50
)

# 5. Train the model
print("\nStarting XGBoost training...")
clf.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50
)

# 6. Evaluate the model
print("\nPredicting on test set...")
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("\n--- Evaluation Results ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"AUC: {roc_auc_score(y_test, y_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Bonafide (0)', 'Spoof (1)']))

Loading extracted features and labels...
Dataset Shape: X=(25380, 512), y=(25380,)
Class 0 (Bonafide): 2580
Class 1 (Spoof): 22800

Starting XGBoost training...
[0]	validation_0-auc:0.95043	validation_1-auc:0.92209
[50]	validation_0-auc:0.99890	validation_1-auc:0.99031
[100]	validation_0-auc:0.99990	validation_1-auc:0.99508
[150]	validation_0-auc:1.00000	validation_1-auc:0.99683
[200]	validation_0-auc:1.00000	validation_1-auc:0.99758
[250]	validation_0-auc:1.00000	validation_1-auc:0.99805
[300]	validation_0-auc:1.00000	validation_1-auc:0.99830
[350]	validation_0-auc:1.00000	validation_1-auc:0.99845
[400]	validation_0-auc:1.00000	validation_1-auc:0.99858
[450]	validation_0-auc:1.00000	validation_1-auc:0.99868
[500]	validation_0-auc:1.00000	validation_1-auc:0.99872
[550]	validation_0-auc:1.00000	validation_1-auc:0.99878
[600]	validation_0-auc:1.00000	validation_1-auc:0.99881
[650]	validation_0-auc:1.00000	validation_1-auc:0.99886
[700]	validation_0-auc:1.00000	validation_1-auc:0.99890
[7

In [6]:
import os
import gc
import torch
import librosa
import numpy as np
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperModel
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# Clear memory safely before starting the massive eval extraction
def free_memory():
    globals_to_clear = ['X_dev', 'y_dev', 'features_list', 'labels_list']
    for var in globals_to_clear:
        if var in globals():
            del globals()[var]

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

free_memory()

# Setup paths for the EVAL dataset
DATASET_ROOT = awsaf49_asvpoof_2019_dataset_path + "/LA/LA"
EVAL_PROTOCOL = os.path.join(DATASET_ROOT, 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019.LA.cm.eval.trl.txt')
EVAL_AUDIO_DIR = os.path.join(DATASET_ROOT, 'ASVspoof2019_LA_eval', 'flac')

OUTPUT_EVAL_FEATURES = 'X_eval_whisper.npy'
OUTPUT_EVAL_LABELS = 'y_eval_whisper.npy'

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing on: {device.upper()}")

# Load Whisper model and processor
model_name = "openai/whisper-base"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperModel.from_pretrained(model_name).to(device)
model.eval()

# Parse the EVAL protocol file
labels_dict = {}
with open(EVAL_PROTOCOL, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 5:
            file_name = parts[1]
            label_str = parts[4]
            labels_dict[file_name] = 0 if label_str == 'bonafide' else 1

print(f"Loaded {len(labels_dict)} annotations from the evaluation protocol.")

# Extract features
features_list = []
labels_list = []
file_names_to_process = list(labels_dict.keys())

print("Starting deep feature extraction for the EVAL dataset...")
for idx, file_name in enumerate(tqdm(file_names_to_process)):
    audio_path = os.path.join(EVAL_AUDIO_DIR, f"{file_name}.flac")

    if not os.path.exists(audio_path):
        continue

    # Load and process audio
    audio_array, sampling_rate = librosa.load(audio_path, sr=16000)
    inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    # Forward pass through encoder only
    with torch.no_grad():
        encoder_outputs = model.encoder(input_features)
        hidden_states = encoder_outputs.last_hidden_state
        audio_embedding = hidden_states.mean(dim=1).squeeze().cpu().numpy()

    features_list.append(audio_embedding)
    labels_list.append(labels_dict[file_name])

    # Clean up local variables explicitly to prevent memory leaks
    del input_features, encoder_outputs, hidden_states, inputs, audio_array

    # Dynamic memory flush every 500 iterations
    if (idx + 1) % 500 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Save extracted features to disk
X_eval = np.array(features_list)
y_eval = np.array(labels_list)

np.save(OUTPUT_EVAL_FEATURES, X_eval)
np.save(OUTPUT_EVAL_LABELS, y_eval)

print(f"\nSaved {X_eval.shape[0]} feature vectors successfully.")

# Clean up extraction artifacts before evaluation
features_list.clear()
labels_list.clear()
free_memory()

# Evaluate the previously trained XGBoost model on the unseen EVAL data
print("\n--- Final Confrontation: Evaluating XGBoost on Unseen EVAL Attacks ---")
try:
    y_pred_eval = clf.predict(X_eval)
    y_proba_eval = clf.predict_proba(X_eval)[:, 1]

    print(f"EVAL Accuracy: {accuracy_score(y_eval, y_pred_eval):.4f}")
    print(f"EVAL AUC: {roc_auc_score(y_eval, y_proba_eval):.4f}")
    print("\nEVAL Classification Report:")
    print(classification_report(y_eval, y_pred_eval, target_names=['Bonafide (0)', 'Spoof (1)']))
except NameError:
    print("\nError: The XGBoost model 'clf' is not found in memory. Please ensure the training cell was executed.")

Executing on: CUDA


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Loaded 71237 annotations from the evaluation protocol.
Starting deep feature extraction for the EVAL dataset...


100%|██████████| 71237/71237 [51:20<00:00, 23.13it/s]



Saved 71237 feature vectors successfully.

--- Final Confrontation: Evaluating XGBoost on Unseen EVAL Attacks ---
EVAL Accuracy: 0.9606
EVAL AUC: 0.9886

EVAL Classification Report:
              precision    recall  f1-score   support

Bonafide (0)       0.76      0.91      0.83      7355
   Spoof (1)       0.99      0.97      0.98     63882

    accuracy                           0.96     71237
   macro avg       0.87      0.94      0.90     71237
weighted avg       0.97      0.96      0.96     71237



In [7]:
import os
import gc
import torch
import librosa
import numpy as np
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperModel
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

def free_memory():
    globals_to_clear = ['X', 'y', 'X_train', 'X_test', 'y_train', 'y_test', 'features_list', 'labels_list']
    for var in globals_to_clear:
        if var in globals():
            del globals()[var]

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

free_memory()

DATASET_ROOT = awsaf49_asvpoof_2019_dataset_path + "/LA/LA"
DEV_PROTOCOL = os.path.join(DATASET_ROOT, 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019.LA.cm.dev.trl.txt')
DEV_AUDIO_DIR = os.path.join(DATASET_ROOT, 'ASVspoof2019_LA_dev', 'flac')

OUTPUT_DEV_FEATURES = 'X_dev_whisper.npy'
OUTPUT_DEV_LABELS = 'y_dev_whisper.npy'

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "openai/whisper-base"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperModel.from_pretrained(model_name).to(device)
model.eval()

labels_dict = {}
with open(DEV_PROTOCOL, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 5:
            file_name = parts[1]
            label_str = parts[4]
            labels_dict[file_name] = 0 if label_str == 'bonafide' else 1

features_list = []
labels_list = []
file_names_to_process = list(labels_dict.keys())

for idx, file_name in enumerate(tqdm(file_names_to_process)):
    audio_path = os.path.join(DEV_AUDIO_DIR, f"{file_name}.flac")

    if not os.path.exists(audio_path):
        continue

    audio_array, sampling_rate = librosa.load(audio_path, sr=16000)
    inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    with torch.no_grad():
        encoder_outputs = model.encoder(input_features)
        hidden_states = encoder_outputs.last_hidden_state
        audio_embedding = hidden_states.mean(dim=1).squeeze().cpu().numpy()

    features_list.append(audio_embedding)
    labels_list.append(labels_dict[file_name])

    del input_features, encoder_outputs, hidden_states, inputs, audio_array

    if (idx + 1) % 500 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

X_dev = np.array(features_list)
y_dev = np.array(labels_list)

np.save(OUTPUT_DEV_FEATURES, X_dev)
np.save(OUTPUT_DEV_LABELS, y_dev)

features_list.clear()
labels_list.clear()
free_memory()

print("\nEvaluating XGBoost on DEV dataset...")
try:
    y_pred_dev = clf.predict(X_dev)
    y_proba_dev = clf.predict_proba(X_dev)[:, 1]

    print(f"Accuracy: {accuracy_score(y_dev, y_pred_dev):.4f}")
    print(f"AUC: {roc_auc_score(y_dev, y_proba_dev):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_dev, y_pred_dev, target_names=['Bonafide (0)', 'Spoof (1)']))
except NameError:
    print("\nError: The XGBoost model 'clf' is not found in memory.")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

100%|██████████| 24844/24844 [17:51<00:00, 23.19it/s]



Evaluating XGBoost on DEV dataset...
Accuracy: 0.9838
AUC: 0.9976

Classification Report:
              precision    recall  f1-score   support

Bonafide (0)       0.98      0.86      0.92      2548
   Spoof (1)       0.98      1.00      0.99     22296

    accuracy                           0.98     24844
   macro avg       0.98      0.93      0.95     24844
weighted avg       0.98      0.98      0.98     24844

